#STEP 1: Install Libraries

In [23]:
import os

# Uninstall and reinstall bitsandbytes and accelerate for Colab compatibility
# This often helps resolve persistent ImportError issues
!pip install -U transformers datasets peft trl
!pip uninstall -y bitsandbytes accelerate

# Install latest versions of bitsandbytes and accelerate
# Check for GPU type for specific bitsandbytes installation if needed
# For general cases, a simple reinstall with -U should suffice
!pip install -U bitsandbytes accelerate

Found existing installation: bitsandbytes 0.49.2
Uninstalling bitsandbytes-0.49.2:
  Successfully uninstalled bitsandbytes-0.49.2
Found existing installation: accelerate 1.13.0
Uninstalling accelerate-1.13.0:
  Successfully uninstalled accelerate-1.13.0
  Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl.metadata (10 kB)
Using cached bitsandbytes-0.49.2-py3-none-manylinux_2_24_x86_64.whl (60.7 MB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 31.8 MB/s eta 0:00:00


#STEP 2: Import Libraries

In [1]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, TaskType

#STEP 3: Load Dataset

In [2]:
dataset = load_dataset("Amod/mental_health_counseling_conversations", split="train")


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


In [17]:
print(dataset[0])

{'Context': "I'm going through some things with my feelings and myself. I barely sleep and I do nothing but think about how I'm worthless and how I shouldn't be here.\n   I've never tried or contemplated suicide. I've always wanted to fix my issues, but I never get around to it.\n   How can I change my feeling of being worthless to everyone?", 'Response': "If everyone thinks you're worthless, then maybe you need to find new people to hang out with.Seriously, the social context in which a person lives is a big influence in self-esteem.Otherwise, you can go round and round trying to understand why you're not worthless, then go back to the same crowd and be knocked down again.There are many inspirational messages you can find in social media. \xa0Maybe read some of the ones which state that no person is worthless, and that everyone has a good purpose to their life.Also, since our culture is so saturated with the belief that if someone doesn't feel good about themselves that this is someho

#STEP 4: Convert to Instruction Format

In [3]:
def format_instruction(example):
    return {
        "text": f"### Instruction:\n{example['Context']}\n\n### Response:\n{example['Response']}"
    }

dataset = dataset.map(format_instruction)

#STEP 5: Load Tokenizer

In [4]:
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"

tokenizer = AutoTokenizer.from_pretrained(model_name)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

#STEP 6: Tokenization

In [5]:
def tokenize_fn(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

dataset = dataset.map(tokenize_fn)

Map:   0%|          | 0/3512 [00:00<?, ? examples/s]

#STEP 7: Load Model (8-bit for low RAM)

In [6]:
bnb_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors:   0%|          | 0.00/4.40G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

#STEP 8: Apply LoRA (PEFT)

In [7]:
lora_config = LoraConfig(
    task_type=TaskType.CAUSAL_LM,
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none"
)

model = get_peft_model(model, lora_config)

#STEP 9: Training Arguments

In [8]:
training_args = TrainingArguments(
    output_dir="./tinyllama-instruct",
    num_train_epochs=3,
    per_device_train_batch_size=1,
    gradient_accumulation_steps=8,
    learning_rate=2e-4,
    fp16=True,
    logging_steps=20,
    save_total_limit=1,
    report_to="none"
)

#STEP 10: Trainer

In [9]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=dataset
)

#STEP 11: Train

In [10]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


Step,Training Loss
20,4.294081
40,1.531289
60,1.387296
80,1.339129
100,1.414301
120,1.360138
140,1.293350
160,1.246798
180,1.362563
200,1.314909


/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")
/usr/local/lib/python3.12/dist-packages/bitsandbytes/autograd/_functions.py:123: UserWarning: MatMul8bitLt: inputs will be cast from torch.float32 to float16 during quantization
  warnings.warn(f"MatMul8bitLt: inputs will be cast from {A.dtype} to float16 during quantization")


TrainOutput(global_step=1317, training_loss=1.3275153088406713, metrics={'train_runtime': 5411.0252, 'train_samples_per_second': 1.947, 'train_steps_per_second': 0.243, 'total_flos': 3.352009798110413e+16, 'train_loss': 1.3275153088406713, 'epoch': 3.0})

#STEP 12: Save LoRA Model

In [11]:
model.save_pretrained("./tinyllama-instruct")
tokenizer.save_pretrained("./tinyllama-instruct")

('./tinyllama-instruct/tokenizer_config.json',
 './tinyllama-instruct/tokenizer.json')

#STEP 13: Load Model (Correct Way)

In [12]:
from peft import PeftModel

base_model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto"
)

model = PeftModel.from_pretrained(base_model, "./tinyllama-instruct")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

#STEP 14: Inference (Chat Style)

In [13]:
prompt = "### Instruction:\nI feel anxious and stressed.\n\n### Response:\n"

inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

output = model.generate(
    **inputs,
    max_new_tokens=100,
    temperature=0.7,
    top_p=0.9
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Both `max_new_tokens` (=100) and `max_length`(=2048) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


### Instruction:
I feel anxious and stressed.

### Response:
I'm sorry to hear that you feel this way.  I'm not sure what you mean by "stressed."  I'm guessing that you mean that you feel anxious.  If so, I would suggest that you try to identify what is causing you to feel anxious.  If you can't identify what is causing you to feel anxious, then you will have a hard time getting rid of it.  If you can identify what is causing you to feel anxious, then
